In [9]:
# ============================================================
# D1 — Branch C: Deterministic Normalisation
# 0. Imports and frozen experimental configuration
# ============================================================
import json, hashlib, math, re, unicodedata
import numpy as np
import openpyxl
import pandas as pd

from pathlib import Path
from datetime import datetime
from google.colab import files

DOCUMENT_ID = "D1"
DOCUMENT_NAME = "CEDEFOP Labour Skills Shortage Dataset"
BRANCH_ID = "C"
BRANCH_NAME = "Deterministic Normalisation"
PARENT_BRANCH = "B"

EXPECTED_SOURCE_FORMAT = ".xlsx"
EXPECTED_SOURCE_SHA256 = "4a0b8117c9abdaa0daeb002455fdda840f6149bf1f68096d33fa4744b975e389"

EXPECTED_SHEETS = ["EU27", "IT", "NL", "PT"]
EXPECTED_RECORD_COUNT = 156

EXPECTED_RECORD_FIELDS = [
    "Geographic Area",
    "Main Occupation Group",
    "Occupation Group (2 digit)",
    "Labour Shortage Index",
    "LSI (Comp.)",
    "LSI1",
    "LSI2",
    "LSI3"
]

NUMERIC_FIELDS = ["Labour Shortage Index", "LSI1", "LSI2", "LSI3"]

BRANCH_B_STRUCTURAL_COLUMNS = [
    "Geographic Area",
    "Main Occupation Group",
    "Occupation Group (2 digit)",
    "Labour Shortage Indexx",
    "LSI (Comp.)",
    "LSI1",
    "LSI2",
    "LSI3",
    "_source_row"
]

OUTPUT_DIR = Path("outputs_D1_branch_C")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Configured D1 Branch C.")


Configured D1 Branch C.


In [2]:
# ------------------------------------------------------------
# 1. Upload required inputs
# ------------------------------------------------------------
# Upload exactly:
# 1) original D1 XLSX
# 2) D1_branch_B_converted_data_audit.csv
# 3) D1_branch_B_conversion_check.json

uploaded = files.upload()
names = list(uploaded.keys())

xlsx = [f for f in names if f.lower().endswith(".xlsx")]
csvs = [f for f in names if f.lower().endswith(".csv")]
jsons = [f for f in names if f.lower().endswith(".json")]

if len(xlsx) != 1 or len(csvs) != 1 or len(jsons) != 1:
    raise ValueError("Upload exactly one XLSX, one Branch B audit CSV, and one Branch B conversion-check JSON.")

SOURCE_PATH = Path(xlsx[0])
BRANCH_B_AUDIT_PATH = Path(csvs[0])
BRANCH_B_CHECK_PATH = Path(jsons[0])

print(SOURCE_PATH.name)
print(BRANCH_B_AUDIT_PATH.name)
print(BRANCH_B_CHECK_PATH.name)


Saving D1_branch_B_converted_data_audit.csv to D1_branch_B_converted_data_audit.csv
Saving D1_branch_B_conversion_check.json to D1_branch_B_conversion_check.json
Saving D1 - 2024_cedefop_labour_skills_shortage_index_clssi_dataset2.xlsx to D1 - 2024_cedefop_labour_skills_shortage_index_clssi_dataset2.xlsx
D1 - 2024_cedefop_labour_skills_shortage_index_clssi_dataset2.xlsx
D1_branch_B_converted_data_audit.csv
D1_branch_B_conversion_check.json


In [3]:
# ------------------------------------------------------------
# 2. Verify frozen source identity and Branch B parent integrity
# ------------------------------------------------------------
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

if SOURCE_PATH.suffix.lower() != EXPECTED_SOURCE_FORMAT:
    raise ValueError("Unexpected source format.")

SOURCE_SHA256 = sha256_file(SOURCE_PATH)
SOURCE_HASH_MATCH = SOURCE_SHA256 == EXPECTED_SOURCE_SHA256

if not SOURCE_HASH_MATCH:
    raise ValueError("Uploaded D1 source does not match the frozen experimental source.")

with open(BRANCH_B_CHECK_PATH, "r", encoding="utf-8") as f:
    branch_b_check = json.load(f)

if branch_b_check.get("document_id") != DOCUMENT_ID or branch_b_check.get("branch") != "B":
    raise ValueError("Uploaded conversion check is not D1 Branch B.")

if not branch_b_check.get("conversion_integrity_passed", False):
    raise ValueError("Branch B parent conversion did not pass integrity checks.")

if branch_b_check.get("source_sha256") != SOURCE_SHA256:
    raise ValueError("Branch B parent was generated from a different source identity.")

print("Source and Branch B parent verified.")


Source and Branch B parent verified.


In [4]:
# ------------------------------------------------------------
# 3. Reproduce the exact Branch B structural row state
# ------------------------------------------------------------
workbook = openpyxl.load_workbook(SOURCE_PATH, data_only=True)

if workbook.sheetnames != EXPECTED_SHEETS:
    raise ValueError(f"Unexpected worksheet order: {workbook.sheetnames}")

def merged_anchor_value(ws, row, col):
    cell = ws.cell(row=row, column=col)
    if cell.value is not None:
        return cell.value
    coordinate = cell.coordinate
    for merged_range in ws.merged_cells.ranges:
        if coordinate in merged_range:
            return ws.cell(row=merged_range.min_row, column=merged_range.min_col).value
    return None

rows = []
for ws in workbook.worksheets:
    for row_idx in range(2, ws.max_row + 1):
        raw_values = [ws.cell(row=row_idx, column=c).value for c in range(1, ws.max_column + 1)]
        if all(v is None for v in raw_values):
            continue
        rows.append({
            "Geographic Area": ws.title,
            "Main Occupation Group": merged_anchor_value(ws, row_idx, 1),
            "Occupation Group (2 digit)": ws.cell(row=row_idx, column=2).value,
            "Labour Shortage Indexx": ws.cell(row=row_idx, column=3).value,
            "LSI (Comp.)": ws.cell(row=row_idx, column=4).value,
            "LSI1": ws.cell(row=row_idx, column=5).value,
            "LSI2": ws.cell(row=row_idx, column=6).value,
            "LSI3": ws.cell(row=row_idx, column=7).value,
            "_source_row": row_idx
        })

branch_b_reproduced_df = pd.DataFrame(rows)[BRANCH_B_STRUCTURAL_COLUMNS]

if len(branch_b_reproduced_df) != EXPECTED_RECORD_COUNT:
    raise ValueError("Reproduced Branch B state has an unexpected record count.")

print("Reproduced Branch B records:", len(branch_b_reproduced_df))


Reproduced Branch B records: 156


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [5]:
# ------------------------------------------------------------
# 4. Verify equivalence with the saved Branch B audit artefact
# ------------------------------------------------------------
saved_b = pd.read_csv(BRANCH_B_AUDIT_PATH)

missing_cols = [c for c in BRANCH_B_STRUCTURAL_COLUMNS if c not in saved_b.columns]
if missing_cols:
    raise ValueError(f"Branch B audit is missing columns: {missing_cols}")

saved_b = saved_b[BRANCH_B_STRUCTURAL_COLUMNS].copy()

def comparable(v):
    if pd.isna(v):
        return None
    if isinstance(v, (np.integer, int)):
        return int(v)
    if isinstance(v, (np.floating, float)):
        return float(v)
    return str(v)

differences = []
for i in range(len(branch_b_reproduced_df)):
    for col in BRANCH_B_STRUCTURAL_COLUMNS:
        a = comparable(branch_b_reproduced_df.iloc[i][col])
        b = comparable(saved_b.iloc[i][col])
        equal = float(a) == float(b) if isinstance(a, (int,float)) and isinstance(b, (int,float)) else a == b
        if not equal:
            differences.append({"row_index": i, "field": col, "reproduced": a, "saved_B": b})

PARENT_EQUIVALENCE_PASSED = len(differences) == 0

parent_check = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,
    "parent_branch": PARENT_BRANCH,
    "source_sha256": SOURCE_SHA256,
    "saved_branch_B_record_count": int(len(saved_b)),
    "reproduced_branch_B_record_count": int(len(branch_b_reproduced_df)),
    "cell_differences_detected": int(len(differences)),
    "differences": differences[:100],
    "parent_equivalence_passed": PARENT_EQUIVALENCE_PASSED
}

PARENT_CHECK_PATH = OUTPUT_DIR / "D1_branch_C_parent_B_equivalence_check.json"
PARENT_CHECK_PATH.write_text(json.dumps(parent_check, indent=2, ensure_ascii=False), encoding="utf-8")

print(json.dumps(parent_check, indent=2, ensure_ascii=False))

if not PARENT_EQUIVALENCE_PASSED:
    raise ValueError("Branch C does not start from the same structural state as Branch B.")


{
  "document_id": "D1",
  "branch": "C",
  "parent_branch": "B",
  "source_sha256": "4a0b8117c9abdaa0daeb002455fdda840f6149bf1f68096d33fa4744b975e389",
  "saved_branch_B_record_count": 156,
  "reproduced_branch_B_record_count": 156,
  "cell_differences_detected": 0,
  "differences": [],
  "parent_equivalence_passed": true
}


In [6]:
# ------------------------------------------------------------
# 5. Define deterministic Branch C normalisation functions
# ------------------------------------------------------------
def normalise_text(value):
    if pd.isna(value):
        return None
    value = unicodedata.normalize("NFKC", str(value))
    replacements = {
        "\u00a0":" ",
        "\u2018":"'", "\u2019":"'",
        "\u201c":'"', "\u201d":'"',
        "\u2010":"-", "\u2011":"-", "\u2012":"-",
        "\u2013":"-", "\u2014":"-", "\u2212":"-"
    }
    for old, new in replacements.items():
        value = value.replace(old, new)
    value = re.sub(r"\s+", " ", value).strip()
    return None if value == "" else value

def normalise_lsi_component(value):
    value = normalise_text(value)
    if value is None:
        return None
    value = value.replace("/", "-").replace("\\", "-").replace("_", "-")
    value = re.sub(r"\s*-\s*", "-", value)
    value = re.sub(r"-+", "-", value)
    return value.strip("-")

def normalise_numeric(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = normalise_text(value)
    if text is None:
        return np.nan
    if "," in text and "." not in text:
        text = text.replace(",", ".")
    try:
        return float(text)
    except (TypeError, ValueError):
        return np.nan

def normalise_integer_indicator(value):
    number = normalise_numeric(value)
    if pd.isna(number):
        return pd.NA
    if float(number).is_integer():
        return int(number)
    return pd.NA


In [7]:
# ------------------------------------------------------------
# 6. Apply only the incremental Branch C normalisation
# ------------------------------------------------------------
normalised_df = branch_b_reproduced_df.copy()

# Canonical header correction only; values remain source-grounded.
normalised_df = normalised_df.rename(columns={
    "Labour Shortage Indexx": "Labour Shortage Index"
})

for col in ["Geographic Area", "Main Occupation Group", "Occupation Group (2 digit)"]:
    normalised_df[col] = normalised_df[col].apply(normalise_text)

normalised_df["LSI (Comp.)"] = normalised_df["LSI (Comp.)"].apply(normalise_lsi_component)
normalised_df["Labour Shortage Index"] = normalised_df["Labour Shortage Index"].apply(normalise_numeric)

for col in ["LSI1", "LSI2", "LSI3"]:
    normalised_df[col] = normalised_df[col].apply(normalise_integer_indicator).astype("Int64")

final_normalised_df = normalised_df[EXPECTED_RECORD_FIELDS].copy()

print("Normalised records:", len(final_normalised_df))
display(final_normalised_df.head())


Normalised records: 156


,Geographic Area,Main Occupation Group,Occupation Group (2 digit),Labour Shortage Index,LSI (Comp.),LSI1,LSI2,LSI3
0,EU27,High-skilled non-manual occupations,"Chief executives, senior officials and legisla...",3.000000,4-4-1,4,4,1
1,EU27,High-skilled non-manual occupations,Administrative and commercial managers,2.333333,4-2-1,4,2,1
2,EU27,High-skilled non-manual occupations,Production and specialised services managers,3.000000,3-4-2,3,4,2
3,EU27,High-skilled non-manual occupations,"Hospitality, retail and other services managers",2.666667,2-3-3,2,3,3
4,EU27,High-skilled non-manual occupations,Science and engineering professionals,2.666667,4-3-1,4,3,1


In [11]:
# ------------------------------------------------------------
# 7. Verify Branch C normalisation integrity
# ------------------------------------------------------------

duplicate_keys = int(
    final_normalised_df.duplicated(
        subset=["Geographic Area", "Occupation Group (2 digit)"]
    ).sum()
)

areas = (
    final_normalised_df["Geographic Area"]
    .dropna()
    .unique()
    .tolist()
)

lsi_pattern = re.compile(r"^[1-4]-[1-4]-[1-4]$")

invalid_lsi = final_normalised_df[
    ~final_normalised_df["LSI (Comp.)"].apply(
        lambda x: False
        if pd.isna(x)
        else bool(lsi_pattern.fullmatch(str(x)))
    )
].copy()


def expected_component(row):
    vals = [
        row["LSI1"],
        row["LSI2"],
        row["LSI3"]
    ]

    if any(pd.isna(v) for v in vals):
        return None

    return "-".join(
        str(int(v))
        for v in vals
    )


expected_components = final_normalised_df.apply(
    expected_component,
    axis=1
)

inconsistent_lsi = final_normalised_df[
    final_normalised_df["LSI (Comp.)"]
    != expected_components
].copy()


# ------------------------------------------------------------
# Observation identity verification
# ------------------------------------------------------------
#
# Branch C is allowed to normalise representation.
#
# Therefore, comparing Branch B and Branch C identity fields using
# raw string equality would be too strict.
#
# Example:
#
# Branch B:
#     "21 – Science and engineering professionals"
#
# Branch C:
#     "21 - Science and engineering professionals"
#
# These are the SAME observation even though the punctuation differs.
#
# We therefore verify observation preservation using:
#
# 1. source worksheet + original source row
# 2. the document-specific observation key after applying the SAME
#    deterministic text normalisation to both Branch B and Branch C
#
# This verifies that Branch C changes representation only and does
# not add, remove, reorder or replace observations.
# ------------------------------------------------------------


# 1. Source provenance / row order must remain exactly unchanged

parent_source_identity = list(
    zip(
        branch_b_reproduced_df[
            "Geographic Area"
        ].astype(str),

        branch_b_reproduced_df[
            "_source_row"
        ].astype(int)
    )
)

branch_c_source_identity = list(
    zip(
        normalised_df[
            "Geographic Area"
        ].astype(str),

        normalised_df[
            "_source_row"
        ].astype(int)
    )
)

source_row_identity_preserved = (
    parent_source_identity
    == branch_c_source_identity
)


# 2. Observation key must remain equivalent after deterministic
#    Branch C text normalisation

def canonical_identity_text(value):
    value = normalise_text(value)

    if value is None:
        return ""

    return value


parent_normalised_keys = list(
    zip(
        branch_b_reproduced_df[
            "Geographic Area"
        ].apply(canonical_identity_text),

        branch_b_reproduced_df[
            "Occupation Group (2 digit)"
        ].apply(canonical_identity_text)
    )
)

branch_c_normalised_keys = list(
    zip(
        final_normalised_df[
            "Geographic Area"
        ].apply(canonical_identity_text),

        final_normalised_df[
            "Occupation Group (2 digit)"
        ].apply(canonical_identity_text)
    )
)

normalised_observation_key_preserved = (
    parent_normalised_keys
    == branch_c_normalised_keys
)


# Both checks must pass

observation_identity_preserved = bool(
    source_row_identity_preserved
    and normalised_observation_key_preserved
)


# ------------------------------------------------------------
# Final Branch C normalisation integrity summary
# ------------------------------------------------------------

normalisation_check = {

    "document_id": DOCUMENT_ID,

    "branch": BRANCH_ID,

    "parent_branch": PARENT_BRANCH,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "normalised_record_count":
        int(len(final_normalised_df)),

    "record_count_preserved":
        len(final_normalised_df)
        == EXPECTED_RECORD_COUNT,

    # Observation preservation
    "source_row_identity_and_order_preserved":
        source_row_identity_preserved,

    "normalised_observation_key_preserved":
        normalised_observation_key_preserved,

    "observation_order_and_identity_preserved":
        observation_identity_preserved,

    # Geographic areas
    "geographic_areas":
        areas,

    "geographic_area_set_preserved":
        areas == EXPECTED_SHEETS,

    # Structural integrity
    "duplicate_observation_keys":
        duplicate_keys,

    "invalid_lsi_component_formats":
        int(len(invalid_lsi)),

    "inconsistent_lsi_components":
        int(len(inconsistent_lsi)),

    # Branch C transformations
    "header_standardisation_applied":
        True,

    "unicode_normalisation_applied":
        True,

    "whitespace_normalisation_applied":
        True,

    "dash_and_separator_normalisation_applied":
        True,

    "numeric_type_normalisation_applied":
        True,

    # Explicitly prohibited Branch C operations
    "semantic_label_mapping_applied":
        False,

    "value_rounding_applied":
        False,

    "derived_calculation_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    # Overall integrity decision
    "normalisation_integrity_passed": bool(

        len(final_normalised_df)
        == EXPECTED_RECORD_COUNT

        and observation_identity_preserved

        and areas
        == EXPECTED_SHEETS

        and duplicate_keys
        == 0

        and len(invalid_lsi)
        == 0

        and len(inconsistent_lsi)
        == 0
    )
}


# ------------------------------------------------------------
# Save integrity check
# ------------------------------------------------------------

NORMALISATION_CHECK_PATH = (
    OUTPUT_DIR
    / "D1_branch_C_normalisation_check.json"
)

NORMALISATION_CHECK_PATH.write_text(
    json.dumps(
        normalisation_check,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print(
    json.dumps(
        normalisation_check,
        indent=2,
        ensure_ascii=False
    )
)


# ------------------------------------------------------------
# Stop execution if Branch C integrity fails
# ------------------------------------------------------------

if not normalisation_check[
    "normalisation_integrity_passed"
]:
    raise ValueError(
        "Branch C normalisation-integrity checks failed. "
        "Inspect the identity diagnostics above."
    )

{
  "document_id": "D1",
  "branch": "C",
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "expected_record_count": 156,
  "normalised_record_count": 156,
  "record_count_preserved": true,
  "source_row_identity_and_order_preserved": true,
  "normalised_observation_key_preserved": true,
  "observation_order_and_identity_preserved": true,
  "geographic_areas": [
    "EU27",
    "IT",
    "NL",
    "PT"
  ],
  "geographic_area_set_preserved": true,
  "duplicate_observation_keys": 0,
  "invalid_lsi_component_formats": 0,
  "inconsistent_lsi_components": 0,
  "header_standardisation_applied": true,
  "unicode_normalisation_applied": true,
  "whitespace_normalisation_applied": true,
  "dash_and_separator_normalisation_applied": true,
  "numeric_type_normalisation_applied": true,
  "semantic_label_mapping_applied": false,
  "value_rounding_applied": false,
  "derived_calculation_applied": false,
  "reference_values_used_for_transformation": false,
  "normalisation_integrity_pas

In [12]:
# ------------------------------------------------------------
# 8. Save normalised audit data
# ------------------------------------------------------------
NORMALISED_DATA_PATH = OUTPUT_DIR / "D1_branch_C_normalised_data_audit.csv"
final_normalised_df.to_csv(NORMALISED_DATA_PATH, index=False)

print("Saved:", NORMALISED_DATA_PATH.name)


Saved: D1_branch_C_normalised_data_audit.csv


In [13]:
# ------------------------------------------------------------
# 9. Build the final Branch C Markdown representation
# ------------------------------------------------------------
def md_cell(value):
    if value is None or pd.isna(value):
        return ""
    text = repr(value) if isinstance(value, float) else str(value)
    return text.replace("\\", "\\\\").replace("|", "\\|").replace("\r\n", "<br>").replace("\n", "<br>").replace("\r", "<br>")

def to_markdown_preserving(df, cols):
    header = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join(["---"] * len(cols)) + " |"
    rows = []
    for _, r in df.iterrows():
        rows.append("| " + " | ".join(md_cell(r[c]) for c in cols) + " |")
    return "\n".join([header, sep] + rows)

representation_cols = [
    "Main Occupation Group",
    "Occupation Group (2 digit)",
    "Labour Shortage Index",
    "LSI (Comp.)",
    "LSI1", "LSI2", "LSI3"
]

sections = []
for area in EXPECTED_SHEETS:
    area_df = final_normalised_df[final_normalised_df["Geographic Area"] == area]
    sections.append(
        f"## Geographic Area: {area}\n\n" +
        to_markdown_preserving(area_df, representation_cols)
    )

BRANCH_C_REPRESENTATION = (
    "# CEDEFOP Labour Skills Shortage Index — Normalised Representation\n\n"
    "The Branch B structural representation has been deterministically normalised "
    "to reduce representational variability. Observation identity, order, hierarchy "
    "and source-grounded values are preserved. No values were inferred, calculated, "
    "rounded or semantically corrected.\n\n"
    + "\n\n".join(sections)
)

REPRESENTATION_PATH = OUTPUT_DIR / "D1_branch_C_normalised_representation.md"
REPRESENTATION_PATH.write_text(BRANCH_C_REPRESENTATION, encoding="utf-8")
REPRESENTATION_SHA256 = sha256_file(REPRESENTATION_PATH)

print(BRANCH_C_REPRESENTATION[:3000])
print("\n[Preview truncated]")


# CEDEFOP Labour Skills Shortage Index — Normalised Representation

The Branch B structural representation has been deterministically normalised to reduce representational variability. Observation identity, order, hierarchy and source-grounded values are preserved. No values were inferred, calculated, rounded or semantically corrected.

## Geographic Area: EU27

| Main Occupation Group | Occupation Group (2 digit) | Labour Shortage Index | LSI (Comp.) | LSI1 | LSI2 | LSI3 |
| --- | --- | --- | --- | --- | --- | --- |
| High-skilled non-manual occupations | Chief executives, senior officials and legislators | 3.0 | 4-4-1 | 4 | 4 | 1 |
| High-skilled non-manual occupations | Administrative and commercial managers | 2.3333332538604736 | 4-2-1 | 4 | 2 | 1 |
| High-skilled non-manual occupations | Production and specialised services managers | 3.0 | 3-4-2 | 3 | 4 | 2 |
| High-skilled non-manual occupations | Hospitality, retail and other services managers | 2.6666667461395264 | 2-3-3 | 2 | 

In [14]:
# ------------------------------------------------------------
# 10. Define the controlled extraction schema and prompt
# ------------------------------------------------------------
EXTRACTION_SCHEMA = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,
    "records": [{
        "Geographic Area": None,
        "Main Occupation Group": None,
        "Occupation Group (2 digit)": None,
        "Labour Shortage Index": None,
        "LSI (Comp.)": None,
        "LSI1": None,
        "LSI2": None,
        "LSI3": None
    }]
}

# Intentionally mirrors Branch B instruction strength.
# The expected answer count is NOT disclosed to the model.
EXTRACTION_TASK = '''
You are an information extraction assistant.

Extract the labour shortage information from the attached deterministically
normalised representation of the CEDEFOP Labour Skills Shortage Index workbook.

Process every Geographic Area section in the representation.

Return one record for each occupation-group observation.

For each record, extract:
- Geographic Area
- Main Occupation Group
- Occupation Group (2 digit)
- Labour Shortage Index
- LSI (Comp.)
- LSI1
- LSI2
- LSI3

Extraction rules:
- Extract only information explicitly supported by the provided representation.
- Use the Geographic Area section heading as the Geographic Area.
- Preserve the association between each geographic area, main occupation
  group, two-digit occupation group and corresponding LSI values.
- Preserve LSI (Comp.) exactly as represented.
- Return LSI1, LSI2 and LSI3 as numerical values.
- Return Labour Shortage Index as a numerical value.
- Use null only when a requested value is not available.
- Do not infer, calculate, reconstruct or invent missing values.
- Do not omit repeated Main Occupation Group values from individual records.
- Return only valid JSON.
- Do not include explanations before or after the JSON.
- Keep the exact field names defined in the schema.
'''.strip()

FULL_PROMPT = f'''
{EXTRACTION_TASK}

Expected JSON schema:
{json.dumps(EXTRACTION_SCHEMA, indent=2, ensure_ascii=False)}

The deterministically normalised Markdown representation is attached
as the extraction source.

Return only the JSON object.
'''.strip()

PROMPT_PATH = OUTPUT_DIR / "D1_branch_C_prompt.txt"
PROMPT_PATH.write_text(FULL_PROMPT, encoding="utf-8")

print(FULL_PROMPT)


You are an information extraction assistant.

Extract the labour shortage information from the attached deterministically
normalised representation of the CEDEFOP Labour Skills Shortage Index workbook.

Process every Geographic Area section in the representation.

Return one record for each occupation-group observation.

For each record, extract:
- Geographic Area
- Main Occupation Group
- Occupation Group (2 digit)
- Labour Shortage Index
- LSI (Comp.)
- LSI1
- LSI2
- LSI3

Extraction rules:
- Extract only information explicitly supported by the provided representation.
- Use the Geographic Area section heading as the Geographic Area.
- Preserve the association between each geographic area, main occupation
  group, two-digit occupation group and corresponding LSI values.
- Preserve LSI (Comp.) exactly as represented.
- Return LSI1, LSI2 and LSI3 as numerical values.
- Return Labour Shortage Index as a numerical value.
- Use null only when a requested value is not available.
- Do not i

In [15]:
# ------------------------------------------------------------
# 11. Preserve experiment metadata and pre-extraction checks
# ------------------------------------------------------------
representation_metadata = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,
    "parent_branch": PARENT_BRANCH,
    "source_sha256": SOURCE_SHA256,
    "parent_B_equivalence_passed": PARENT_EQUIVALENCE_PASSED,
    "representation_type": "Deterministically normalised Markdown",
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,
    "structural_conversion_inherited_from_branch_B": True,
    "normalisation_applied": True,
    "semantic_label_mapping_applied": False,
    "value_rounding_applied": False,
    "derived_calculation_applied": False,
    "reference_values_used_for_transformation": False,
    "record_count": int(len(final_normalised_df))
}

REP_METADATA_PATH = OUTPUT_DIR / "D1_branch_C_representation_metadata.json"
REP_METADATA_PATH.write_text(
    json.dumps(representation_metadata, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

experiment_metadata = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH_ID,
    "branch_name": BRANCH_NAME,
    "parent_branch": PARENT_BRANCH,
    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "input_representation": "Deterministically normalised Markdown",
    "parent_B_equivalence_passed": PARENT_EQUIVALENCE_PASSED,
    "normalisation_integrity_passed": normalisation_check["normalisation_integrity_passed"],
    "reference_values_disclosed_to_model": False,
    "expected_record_count_disclosed_to_model": False,
    "manual_response_repair_permitted": False,
    "execution_environment": "ChatGPT independent conversation",
    "validation_status": "Pending Stage 4 Branch C validation using fixed Stage 1 reference values and Branch A-frozen comparison rules"
}

METADATA_PATH = OUTPUT_DIR / "D1_branch_C_experiment_metadata.json"
METADATA_PATH.write_text(
    json.dumps(experiment_metadata, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

precheck = {
    "source_identity_verified": SOURCE_HASH_MATCH,
    "parent_B_equivalence_passed": PARENT_EQUIVALENCE_PASSED,
    "normalisation_integrity_passed": normalisation_check["normalisation_integrity_passed"],
    "representation_exists": REPRESENTATION_PATH.exists(),
    "prompt_exists": PROMPT_PATH.exists(),
    "expected_record_count_disclosed_to_model": False,
    "reference_values_used_for_transformation": False,
    "ready_for_independent_llm_execution": bool(
        SOURCE_HASH_MATCH and
        PARENT_EQUIVALENCE_PASSED and
        normalisation_check["normalisation_integrity_passed"]
    )
}

PRECHECK_PATH = OUTPUT_DIR / "D1_branch_C_pre_extraction_check.json"
PRECHECK_PATH.write_text(
    json.dumps(precheck, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(precheck, indent=2))


{
  "source_identity_verified": true,
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "representation_exists": true,
  "prompt_exists": true,
  "expected_record_count_disclosed_to_model": false,
  "reference_values_used_for_transformation": false,
  "ready_for_independent_llm_execution": true
}


In [16]:
# ------------------------------------------------------------
# 12. Download Branch C model-input and audit artefacts
# ------------------------------------------------------------
for p in [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    NORMALISED_DATA_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    REP_METADATA_PATH,
    METADATA_PATH,
    PRECHECK_PATH
]:
    files.download(p)

print(
    "Independent execution:\n"
    "1. Open a new ChatGPT conversation.\n"
    "2. Upload ONLY D1_branch_C_normalised_representation.md.\n"
    "3. Submit D1_branch_C_prompt.txt exactly once.\n"
    "4. Do not upload the original XLSX, Branch B artefacts, or Stage 1 reference values.\n"
    "5. Do not repair, regenerate, or manually correct the response.\n"
    "6. Save the complete response exactly as returned in a .txt file."
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Independent execution:
1. Open a new ChatGPT conversation.
2. Upload ONLY D1_branch_C_normalised_representation.md.
3. Submit D1_branch_C_prompt.txt exactly once.
4. Do not upload the original XLSX, Branch B artefacts, or Stage 1 reference values.
5. Do not repair, regenerate, or manually correct the response.
6. Save the complete response exactly as returned in a .txt file.


In [17]:
# ------------------------------------------------------------
# 13. Upload and preserve the raw Branch C response
# ------------------------------------------------------------
uploaded_response = files.upload()

if len(uploaded_response) != 1:
    raise ValueError("Upload exactly one raw Branch C response text file.")

RAW_RESPONSE_SOURCE = Path(next(iter(uploaded_response)))
raw_response_text = RAW_RESPONSE_SOURCE.read_text(encoding="utf-8")

RAW_RESPONSE_PATH = OUTPUT_DIR / "D1_branch_C_raw_response.txt"
RAW_RESPONSE_PATH.write_text(raw_response_text, encoding="utf-8")
RAW_RESPONSE_SHA256 = sha256_file(RAW_RESPONSE_PATH)

print("Raw response preserved unchanged.")
print("SHA-256:", RAW_RESPONSE_SHA256)


Saving D1_branch_C_raw_response.txt to D1_branch_C_raw_response.txt
Raw response preserved unchanged.
SHA-256: f7532cfd851cc2f22685776b4d03ba1ecd31c7547c2e5ad301fb1768ca0bce9a


In [18]:
# ------------------------------------------------------------
# 14. Parse raw response without content repair
# ------------------------------------------------------------
json_valid = True
json_error = None
raw_extraction = None

try:
    raw_extraction = json.loads(raw_response_text)
except json.JSONDecodeError as exc:
    json_valid = False
    json_error = str(exc)

print("JSON valid:", json_valid)
if json_error:
    print(json_error)


JSON valid: True


In [19]:
# ------------------------------------------------------------
# 15. Technical/schema/scope diagnostics only
# ------------------------------------------------------------
top_level_checks = {
    "output_is_json_object": isinstance(raw_extraction, dict) if json_valid else False,
    "document_id_present": isinstance(raw_extraction, dict) and "document_id" in raw_extraction if json_valid else False,
    "document_id_correct": isinstance(raw_extraction, dict) and raw_extraction.get("document_id") == DOCUMENT_ID if json_valid else False,
    "branch_present": isinstance(raw_extraction, dict) and "branch" in raw_extraction if json_valid else False,
    "branch_correct": isinstance(raw_extraction, dict) and raw_extraction.get("branch") == BRANCH_ID if json_valid else False,
    "records_present": isinstance(raw_extraction, dict) and "records" in raw_extraction if json_valid else False,
    "records_is_list": isinstance(raw_extraction, dict) and isinstance(raw_extraction.get("records"), list) if json_valid else False
}

records = raw_extraction.get("records", []) if json_valid and isinstance(raw_extraction, dict) else []
observed_record_count = len(records) if isinstance(records, list) else None

record_structure_issues = []
field_type_issues = []

for idx, record in enumerate(records):
    if not isinstance(record, dict):
        record_structure_issues.append({"record_index": idx, "issue": "record_is_not_json_object"})
        continue

    actual = set(record.keys())
    expected = set(EXPECTED_RECORD_FIELDS)
    missing = sorted(expected - actual)
    extra = sorted(actual - expected)

    if missing or extra:
        record_structure_issues.append({
            "record_index": idx,
            "missing_fields": missing,
            "additional_fields": extra
        })

    for field in NUMERIC_FIELDS:
        if field in record:
            value = record[field]
            if value is not None and (isinstance(value, bool) or not isinstance(value, (int, float))):
                field_type_issues.append({
                    "record_index": idx,
                    "field": field,
                    "value": value,
                    "observed_type": type(value).__name__
                })

structure_check = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,
    "json_valid": json_valid,
    "json_error": json_error,
    "top_level_checks": top_level_checks,
    "record_count_check": {
        "expected_record_count": EXPECTED_RECORD_COUNT,
        "observed_record_count": observed_record_count,
        "record_count_matches": observed_record_count == EXPECTED_RECORD_COUNT if observed_record_count is not None else False
    },
    "records_with_structure_issues": len(record_structure_issues),
    "record_structure_issues": record_structure_issues,
    "numeric_field_type_issues": len(field_type_issues),
    "field_type_issues": field_type_issues
}

STRUCTURE_CHECK_PATH = OUTPUT_DIR / "D1_branch_C_structure_check.json"
STRUCTURE_CHECK_PATH.write_text(
    json.dumps(structure_check, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(structure_check, indent=2, ensure_ascii=False))


{
  "document_id": "D1",
  "branch": "C",
  "json_valid": true,
  "json_error": null,
  "top_level_checks": {
    "output_is_json_object": true,
    "document_id_present": true,
    "document_id_correct": true,
    "branch_present": true,
    "branch_correct": true,
    "records_present": true,
    "records_is_list": true
  },
  "record_count_check": {
    "expected_record_count": 156,
    "observed_record_count": 156,
    "record_count_matches": true
  },
  "records_with_structure_issues": 0,
  "record_structure_issues": [],
  "numeric_field_type_issues": 0,
  "field_type_issues": []
}


In [20]:
# ------------------------------------------------------------
# 16. Preserve parsed extraction and experiment summary
# ------------------------------------------------------------
PARSED_EXTRACTION_PATH = OUTPUT_DIR / "D1_branch_C_parsed_extraction.json"

if json_valid:
    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(raw_extraction, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

summary = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,
    "parent_branch": PARENT_BRANCH,
    "source_sha256": SOURCE_SHA256,
    "parent_B_equivalence_passed": PARENT_EQUIVALENCE_PASSED,
    "normalisation_integrity_passed": normalisation_check["normalisation_integrity_passed"],
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,
    "reference_values_used_for_transformation": False,
    "expected_record_count_disclosed_to_model": False,
    "json_valid": json_valid,
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "observed_record_count": observed_record_count,
    "record_count_matches": observed_record_count == EXPECTED_RECORD_COUNT if observed_record_count is not None else False,
    "records_with_structure_issues": len(record_structure_issues),
    "numeric_field_type_issues": len(field_type_issues),
    "raw_response_preserved": True,
    "raw_response_sha256": RAW_RESPONSE_SHA256,
    "parsed_extraction_created": json_valid,
    "validation_status": "Pending Stage 4 Branch C validation against the fixed Stage 1 reference dataset using Branch A-frozen comparison rules"
}

SUMMARY_PATH = OUTPUT_DIR / "D1_branch_C_experiment_summary.json"
SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(summary, indent=2, ensure_ascii=False))


{
  "document_id": "D1",
  "branch": "C",
  "parent_branch": "B",
  "source_sha256": "4a0b8117c9abdaa0daeb002455fdda840f6149bf1f68096d33fa4744b975e389",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "representation_file": "D1_branch_C_normalised_representation.md",
  "representation_sha256": "b6e81848a0f5e21bbb6d8f0daee32c35dd32c769efdce1ebbb3cc06eedfdd33b",
  "reference_values_used_for_transformation": false,
  "expected_record_count_disclosed_to_model": false,
  "json_valid": true,
  "expected_record_count": 156,
  "observed_record_count": 156,
  "record_count_matches": true,
  "records_with_structure_issues": 0,
  "numeric_field_type_issues": 0,
  "raw_response_preserved": true,
  "raw_response_sha256": "f7532cfd851cc2f22685776b4d03ba1ecd31c7547c2e5ad301fb1768ca0bce9a",
  "parsed_extraction_created": true,
  "validation_status": "Pending Stage 4 Branch C validation against the fixed Stage 1 reference dataset using Branch A-frozen comparison rules

In [21]:
# ------------------------------------------------------------
# 17. Final artefact inventory and download
# ------------------------------------------------------------
artefacts = [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    NORMALISED_DATA_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    REP_METADATA_PATH,
    METADATA_PATH,
    PRECHECK_PATH,
    RAW_RESPONSE_PATH,
    STRUCTURE_CHECK_PATH,
    SUMMARY_PATH
]

if json_valid:
    artefacts.append(PARSED_EXTRACTION_PATH)

for p in artefacts:
    print("-", p.name, "| exists:", p.exists())

for p in artefacts:
    if p.exists():
        files.download(p)


- D1_branch_C_parent_B_equivalence_check.json | exists: True
- D1_branch_C_normalisation_check.json | exists: True
- D1_branch_C_normalised_data_audit.csv | exists: True
- D1_branch_C_normalised_representation.md | exists: True
- D1_branch_C_prompt.txt | exists: True
- D1_branch_C_representation_metadata.json | exists: True
- D1_branch_C_experiment_metadata.json | exists: True
- D1_branch_C_pre_extraction_check.json | exists: True
- D1_branch_C_raw_response.txt | exists: True
- D1_branch_C_structure_check.json | exists: True
- D1_branch_C_experiment_summary.json | exists: True
- D1_branch_C_parsed_extraction.json | exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>